In [0]:
dbutils.widgets.removeAll()

In [0]:
%sql
create widget text storageName default "sasmartdata010826ke";

In [0]:
storageName = dbutils.widgets.get("storageName")

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-metastore`
URL 'abfss://metastore-adb@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas raw del Data Lake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-raw`
URL 'abfss://raw@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas raw del Data Lake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-bronze`
URL 'abfss://bronze@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas bronze del Data Lake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-silver`
URL 'abfss://silver@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas silver del Data Lake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-golden`
URL 'abfss://golden@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas golden del Data Lake';

In [0]:
%sql
-- DROP CATALOG IF EXISTS catalog_au CASCADE; -- Descomentar si se requiere recrear todo desde cero

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS catalog_au
MANAGED LOCATION 'abfss://metastore-adb@${storageName}.dfs.core.windows.net/'
COMMENT 'Catalogo para la arquitectura medallion del diccionario de datos';

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS catalog_au.raw;
CREATE SCHEMA IF NOT EXISTS catalog_au.bronze;
CREATE SCHEMA IF NOT EXISTS catalog_au.silver;
CREATE SCHEMA IF NOT EXISTS catalog_au.golden;

### Tablas Bronze

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_au.bronze.conceptos_negocio (
  caso_de_uso STRING,
  codigo_de_dominio STRING,
  nombre_del_dominio STRING,
  codigo_del_subdominio STRING,
  nombre_del_subdominio STRING,
  nombre_del_subdominio_omg STRING,
  concepto_de_negocio STRING,
  termino_de_negocio STRING,
  codigo_de_entidad_del_dato STRING,
  data_owner STRING,
  dato_critico STRING,
  personal STRING,
  sensible STRING,
  definicion_de_negocio STRING,
  tipo_de_dato STRING,
  indicadores_empleados_en_el_calculo STRING,
  logica_de_calculo STRING,
  ejemplo_de_valores_del_entidad_del_dato STRING,
  prioridad_del_entidad_de_dato STRING,
  indicador_dimension STRING,
  periodicidad_de_generacion STRING,
  areas_usuarias_del_dato STRING,
  validado_por_data_owner STRING,
  responsable_de_la_actualizacion_en_el_diccionario_negocio STRING,
  ingestion_date TIMESTAMP
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/conceptos_negocio";

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_au.bronze.reglas_calidad (
  iniciativa STRING,
  caso_de_uso STRING,
  id_regla_de_calidad STRING,
  termino_de_negocio STRING,
  tabla_del_entidad_de_dato_omg STRING,
  campo_del_entidad_de_dato_omg STRING,
  nombre_del_entidad_de_dato STRING,
  descripcion_de_regla_de_calidad STRING,
  tipo_de_regla_de_calidad STRING,
  cumplimiento_regla_de_calidad_core STRING,
  principio_de_calidad_asociado STRING,
  umbral_superior DOUBLE,
  umbral_inferior DOUBLE,
  periodicidad STRING,
  aplicacion STRING,
  data_owner STRING,
  validado_por_data_owner STRING,
  ingestion_date TIMESTAMP
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/reglas_calidad";

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_au.bronze.trazabilidad (
  iniciativa STRING,
  caso_de_uso STRING,
  codigo_de_entidad_del_dato STRING,
  tabla_del_entidad_de_dato_omg STRING,
  campo_del_entidad_de_dato_omg STRING,
  nombre_de_entidad_del_dato STRING,
  llave STRING,
  aplicativos STRING,
  fuente_oficial STRING,
  compania STRING,
  esquema STRING,
  tabla_en_fuente_oficial STRING,
  campo_en_fuente_oficial STRING,
  personal STRING,
  sensible STRING,
  fuentes_y_campos_necesarias_para_el_calculo_o_generacion_del_dato STRING,
  ruta_u_origen_del_repositorio_servidor STRING,
  data_entry STRING,
  periodicidad_de_actualizacion STRING,
  profundidad_de_datos STRING,
  formato_del_dato STRING,
  longitud STRING,
  es_llave_primaria STRING,
  data_owner STRING,
  data_custodian STRING,
  validado_por_data_custodian STRING,
  fecha_de_actualizacion_en_el_diccionario_tecnico STRING,
  actualizacion_realizada_en_el_diccionario_tecnico STRING,
  motivo_de_actualizacion_en_el_diccionario_tecnico STRING,
  responsable_de_la_actualizacion_en_el_diccionario_tecnico STRING,
  ingestion_date TIMESTAMP
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/trazabilidad";

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_au.bronze.app_custom_concepts (
  concepto_de_negocio STRING,
  termino_de_negocio STRING,
  codigo_de_entidad_del_dato STRING,
  data_owner STRING,
  dato_critico STRING,
  personal STRING,
  sensible STRING,
  definicion_de_negocio STRING,
  prioridad_del_entidad_de_dato STRING,
  fecha_registro TIMESTAMP
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/app_custom_concepts";

### Tablas Silver

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_au.silver.conceptos_calidad_trazabilidad (
  codigo_de_entidad_del_dato STRING,
  concepto_de_negocio STRING,
  termino_de_negocio STRING,
  nombre_del_dominio STRING,
  nombre_del_subdominio STRING,
  data_owner STRING,
  dato_critico STRING,
  personal STRING,
  sensible STRING,
  prioridad_del_entidad_de_dato STRING,
  id_regla_de_calidad STRING,
  descripcion_de_regla_de_calidad STRING,
  principio_de_calidad_asociado STRING,
  umbral_superior DOUBLE,
  umbral_inferior DOUBLE,
  tabla_en_fuente_oficial STRING,
  campo_en_fuente_oficial STRING,
  aplicativo STRING,
  fuente_oficial STRING,
  fecha_actualizacion_diccionario TIMESTAMP,
  is_custom_concept BOOLEAN,
  ingestion_date TIMESTAMP
)
USING DELTA
LOCATION "abfss://silver@${storageName}.dfs.core.windows.net/conceptos_calidad_trazabilidad";

### Tablas Golden

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_au.golden.reporte_calidad_conceptos (
  nombre_del_dominio STRING,
  nombre_del_subdominio STRING,
  total_conceptos LONG,
  total_datos_criticos LONG,
  total_datos_personales LONG,
  total_datos_sensibles LONG,
  total_reglas_calidad LONG,
  promedio_umbral_superior DOUBLE,
  promedio_umbral_inferior DOUBLE,
  ultimo_procesamiento TIMESTAMP
)
USING DELTA
LOCATION "abfss://golden@${storageName}.dfs.core.windows.net/reporte_calidad_conceptos";

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_au.golden.resumen_prioridad_dominio (
  nombre_del_dominio STRING,
  prioridad_del_entidad_de_dato STRING,
  cantidad_conceptos LONG,
  ultimo_procesamiento TIMESTAMP
)
USING DELTA
LOCATION "abfss://golden@${storageName}.dfs.core.windows.net/resumen_prioridad_dominio";